In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

import openquantum_sde
from openquantum_sde.integrators import EulerMaruyama, splittingRK4EM
from openquantum_sde.systems import TransmonCavity
from openquantum_sde.simulation import simulate_fixed_dt, simulate_adaptive_dt
from openquantum_sde.utils import calculate_norm, calculate_num_atoms


## Main run

In [ ]:
# Transmon/cavity systems parameters and initial conditions
maxAt = 8 #8 #8 #2 #8 #transmon
maxPh = 250 #250 #400 # 400 #10 #400 #photon
k = 1.0 
Omega, epsilon, U = 50.0*k, 12.0*k, 400.0*k 
X0 = np.zeros([maxAt+1,maxPh+1], dtype=np.complex128)
X0[0,0] = 1.0 

# Define system
M, N = X0.shape
trans_cavity_system = TransmonCavity(M, N, k, Omega, epsilon, U)

# Define integrator
myIntegrator = splittingRK4EM(M,N)

# Run simulation
dt_array, times, traj, traj_current = simulate_adaptive_dt(
    X0 = X0, 
    nsteps_max = 5000000, #50000000, #400000, #5000000, #1000000, 
    dt_min = 1e-6, #1e-6, #0.0005/10.0,
    dt_max = 2e-4, #2e-4, #2e-4, #0.0005*10.0,
    tol = 0.95,
    save_every = 100, 
    calculate_current = True,
    integrator = myIntegrator,
    system = trans_cavity_system
    )

In [ ]:
# Check values of norms
for i in range(len(traj)):
    norm = calculate_norm(traj[i])
    if np.abs(1.0 - norm) >= 0.1:
        print(i, norm)

In [ ]:
# Plot current alpha (real, imag and norm)
fig, ax = plt.subplots(figsize=(6, 4), dpi=120)
ax.plot(times, traj_current.real, label=r'Re[$\alpha$]', lw=0.5)
ax.plot(times, traj_current.imag, label=r'Im[$\alpha$]', lw=0.5)
ax.plot(times, (traj_current*traj_current.conjugate()).real, label=r'$|\alpha^2|$', lw=0.5)
ax.set_xlabel('Time')
ax.legend()
ax.grid(True, alpha=0.3)

In [ ]:
# Plot phase space of current
scale = 1.0
maxval = scale*(abs(epsilon)/k)
fig, ax = plt.subplots(figsize=(6, 6), dpi=120)
ax.plot(traj_current.real, traj_current.imag, lw=0.2, color='k')
ax.set_xlim([-maxval,maxval])
ax.set_ylim([-maxval,maxval])
ax.set_aspect('equal')

In [ ]:
print(times[-1],max(dt_array), min(dt_array))
print(dt_array[-200:-1])

In [ ]:
# Arrays to calculate num atoms
traj_num_atoms = np.zeros(traj.shape[0:2], dtype=np.float64)
for i in range(traj.shape[0]):
    traj_num_atoms[i] = calculate_num_atoms(traj[i])

In [ ]:
# Plot num_atoms histogram (averaged over time interval)
imin = 0
imax = len(traj_num_atoms)
mean_num_atoms = np.mean(traj_num_atoms[imin:imax], axis=0)
plt.bar(np.arange(0, len(mean_num_atoms)), mean_num_atoms)
plt.xlabel("Value")
plt.ylabel("Frequency")